In [1]:
!pip install -q onnx onnxruntime onnxscript
!pip install -q pytorch-msssim
import os, glob, random, json, math
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v2
from pytorch_msssim import ssim
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

ROOT = "/content/data"
os.makedirs(ROOT, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 8.8 MB/s eta 0:00:00
Using device: cuda


In [2]:
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!unzip -q /content/val2017.zip -d /content/coco

COCO_DIR = "/content/coco/val2017"
all_paths = sorted(glob.glob(os.path.join(COCO_DIR, "*.jpg")))
print("Total COCO val2017 images:", len(all_paths))

random.shuffle(all_paths)
clean_paths = all_paths[:2000]

train_paths, temp_paths = train_test_split(clean_paths, test_size=0.30, random_state=SEED)
val_paths, test_paths = train_test_split(temp_paths, test_size=0.50, random_state=SEED)

print(f"train={len(train_paths)} val={len(val_paths)} test={len(test_paths)}")

Total COCO val2017 images: 5000
train=1400 val=300 test=300


In [22]:
def compute_classical_features(img_bgr):
    """img_bgr: HxWx3 uint8 BGR image. Returns an 16-dim np.float32 vector."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # 1. Sharpness — variance of Laplacian
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

    # 2. Brightness — mean pixel intensity
    brightness = gray.mean()

    # 3. Contrast — std dev of intensity
    contrast = gray.std()

    # 4. Noise estimate — high-frequency residual after median blur
    median = cv2.medianBlur(gray, 3)
    noise_estimate = np.mean(np.abs(gray.astype(np.float32) - median.astype(np.float32)))

    # 5. Saturation mean (HSV channel 1)
    saturation = hsv[:, :, 1].mean()

    # 6. Edge density — fraction of Canny edge pixels
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.mean(edges > 0)

    # 7. Entropy — histogram-based
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    hist = hist / (hist.sum() + 1e-8)
    entropy = -np.sum(hist * np.log2(hist + 1e-8))

    # 8. Colorfulness (Hasler-Süsstrunk metric)
    b, g, r = cv2.split(img_bgr.astype(np.float32))
    rg = r - g
    yb = 0.5 * (r + g) - b
    colorfulness = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2)

    pct_shadow_clip_15 = np.mean(gray < 15)
    pct_shadow_clip_30 = np.mean(gray < 30)

    pct_highlight_clip_240 = np.mean(gray > 240)
    pct_highlight_clip_225 = np.mean(gray > 225)

    # Histogram percentiles
    p01 = np.percentile(gray, 1)
    p05 = np.percentile(gray, 5)
    p95 = np.percentile(gray, 95)
    p99 = np.percentile(gray, 99)

    feats = np.array([sharpness, brightness, contrast, noise_estimate,
                       saturation, edge_density, entropy, colorfulness,
                       pct_shadow_clip_15, pct_shadow_clip_30,
                       pct_highlight_clip_240, pct_highlight_clip_225,
                       p01, p05, p95, p99], dtype=np.float32)
    return feats

def resize_with_padding(img, target_size):
    h, w = img.shape[:2]
    scale = target_size / max(h, w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (new_w, new_h))
    pad_h, pad_w = target_size - new_h, target_size - new_w
    top, bottom = pad_h // 2, pad_h - pad_h // 2
    left, right = pad_w // 2, pad_w - pad_w // 2
    return cv2.copyMakeBorder(resized, top, bottom, left, right,
                               cv2.BORDER_CONSTANT, value=[0, 0, 0])

CLASSICAL_FEAT_NAMES = ["sharpness", "brightness", "contrast",
                        "noise_estimate", "saturation", "edge_density",
                        "entropy", "colorfulness", "pct_shadow_clip_15",
                        "pct_shadow_clip_30", "pct_highlight_clip_240",
                        "pct_highlight_clip_225", "p01", "p05", "p95", "p99"
                        ]

In [23]:
def apply_blur(img, sigma):
    k = max(3, int(sigma * 6) | 1)
    return cv2.GaussianBlur(img, (k, k), sigma)

def apply_exposure(img, gamma):
    table = (
        (np.arange(256) / 255.0) ** gamma * 255
    ).astype(np.uint8)

    return cv2.LUT(img, table)

def apply_noise(img, sigma):
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype("uint8")

def apply_corruption(img, quality):
    ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

def apply_synthetic_defect(img):

    out = img.copy()
    h, w = out.shape[:2]
    n_marks = random.randint(1, 3)
    for _ in range(n_marks):
        if random.random() < 0.5:
            x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
            x2, y2 = x1 + random.randint(20, 80), y1 + random.randint(20, 80)
            color = tuple(int(c) for c in np.random.randint(0, 255, 3))
            cv2.rectangle(out, (x1, y1), (x2, y2), color, -1)
        else:
            x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
            x2, y2 = x1 + random.randint(-100, 100), y1 + random.randint(-100, 100)
            cv2.line(out, (x1, y1), (x2, y2), (0, 0, 0), thickness=random.randint(2, 6))
    return out

def apply_cutpaste_defect(img, other_imgs):
    out = img.copy()
    h, w = out.shape[:2]
    donor = random.choice(other_imgs)
    dh, dw = donor.shape[:2]
    patch_h = min(random.randint(int(h*0.15), int(h*0.35)), dh)
    patch_w = min(random.randint(int(w*0.15), int(w*0.35)), dw)
    sy, sx = random.randint(0, dh - patch_h), random.randint(0, dw - patch_w)
    patch = donor[sy:sy+patch_h, sx:sx+patch_w]
    ty, tx = random.randint(0, h - patch_h), random.randint(0, w - patch_w)
    out[ty:ty+patch_h, tx:tx+patch_w] = patch
    return out

def apply_ood_defect(img):
    out = img.copy()
    h, w = out.shape[:2]
    choice = random.choice(["block", "line", "checkerboard", "channel_shift", "local_noise", "scratch", "dead_pixel"])

    if choice == "block":
        x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
        x2, y2 = x1 + random.randint(20, 80), y1 + random.randint(20, 80)
        color = tuple(int(c) for c in np.random.randint(0, 255, 3))
        cv2.rectangle(out, (x1, y1), (x2, y2), color, -1)

    elif choice == "line":
      x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
      x2, y2 = x1 + random.randint(-100, 100), y1 + random.randint(-100, 100)
      cv2.line(out, (x1, y1), (x2, y2), (0, 0, 0), thickness=random.randint(5, 10))

    elif choice == "checkerboard":
        size = random.randint(40, 90)
        x, y = random.randint(0, w - size), random.randint(0, h - size)
        tile = 8
        patch = out[y:y+size, x:x+size].copy()
        for i in range(0, size, tile):
            for j in range(0, size, tile):
                if (i // tile + j // tile) % 2 == 0:
                    patch[i:i+tile, j:j+tile] = [255, 255, 255]
                else:
                    patch[i:i+tile, j:j+tile] = [0, 0, 0]
        out[y:y+size, x:x+size] = patch

    elif choice == "channel_shift":
        shift = random.choice([
            "red",
            "green",
            "blue"
        ])

        amount = random.randint(40, 120)

        if shift == "red":
            # BGR image → channel 2 is red
            out[:, :, 2] = np.clip(
                out[:, :, 2].astype(np.int16) + amount,
                0,
                255
            ).astype(np.uint8)

        elif shift == "green":
            out[:, :, 1] = np.clip(
                out[:, :, 1].astype(np.int16) + amount,
                0,
                255
            ).astype(np.uint8)

        else:
            out[:, :, 0] = np.clip(
                out[:, :, 0].astype(np.int16) + amount,
                0,
                255
            ).astype(np.uint8)

    elif choice == "local_noise":
        size = random.randint(40, 100)
        x, y = random.randint(0, w - size), random.randint(0, h - size)

        patch = out[y : y + size, x : x + size]
        noise = np.random.normal(0, random.uniform(30, 70), patch.shape)
        noisy = np.clip(patch.astype(np.float32) + noise, 0, 255).astype(np.uint8)

        out[y:y+size, x:x+size] = noisy

    elif choice == "scratch":
        x1, y1 = random.randint(0, w - 1), random.randint(0, h - 1)
        length = random.randint(
            max(10, int(min(h, w) * 0.05)),
            max(20, int(min(h, w) * 0.40))
        )
        angle = random.uniform(0, 2 * np.pi)

        x2, y2 = int(x1 + length * np.cos(angle)), int(y1 + length * np.sin(angle))
        x2, y2 = np.clip(x2, 0, w - 1), np.clip(y2, 0, h - 1)
        thickness = random.randint(4, max(6, int(min(h,w)*0.015)))

        if random.random() < 0.5:
            color_value = random.randint(180, 255)
        else:
            color_value = random.randint(0, 70)

        color = (color_value, color_value, color_value)

        cv2.line(out, (x1, y1), (x2, y2), color, thickness)

        if random.random() < 0.35:
            offset = random.randint(2, 8)
            cv2.line(
                out,
                (
                    np.clip(x1 + offset, 0, w - 1),
                    np.clip(y1 + offset, 0, h - 1)
                ),
                (
                    np.clip(x2 + offset, 0, w - 1),
                    np.clip(y2 + offset, 0, h - 1)
                ),
                color,
                max(1, thickness // 2)
            )

    elif choice == "dead_pixel":
        n_pixels = random.randint(20, 60)

        for _ in range(n_pixels):
            x, y = random.randint(0, w - 1), random.randint(0, h - 1)

            defect_type = random.choice([
                "black",
                "white",
                "stuck_color"
            ])

            if defect_type == "black":
                color = (0, 0, 0)
            elif defect_type == "white":
                color = (255, 255, 255)
            else:
                color = tuple(
                    int(v)
                    for v in np.random.randint(0, 256, 3)
                )

            if random.random() < 0.4:
                radius = random.randint(1, 2)
                cv2.circle(
                    out,
                    (x, y),
                    radius,
                    color,
                    -1
                )
            else:
                out[y, x] = color

    else:
        size = random.randint(40, 90)
        x, y = random.randint(0, w - size), random.randint(0, h - size)
        patch = out[y:y+size, x:x+size].copy()
        ch = random.randint(0, 2)
        patch[:, :, ch] = 255 - patch[:, :, ch]
        out[y:y+size, x:x+size] = patch
    return out

ISSUE_NAMES = ["blur", "underexposure", "overexposure", "noise", "corruption"]
ISSUE_PENALTY = {"blur": 25, "underexposure": 20, "overexposure": 20, "noise": 15, "corruption": 30}

def generate_variant(img_bgr):
    label = np.zeros(len(ISSUE_NAMES), dtype=np.float32)
    severities = {}
    out = img_bgr.copy()

    if random.random() < 0.15:
        score = 100.0
        return out, label, severities, score

    possible = random.sample(ISSUE_NAMES, k=random.randint(1, 3))

    for issue in possible:
        if issue == "blur":
            sigma = random.uniform(1.0, 5.0)
            out = apply_blur(out, sigma)
            sev = min(sigma / 5.0, 1.0)
        elif issue == "underexposure":
            gamma = random.uniform(1.8, 3.5)
            out = apply_exposure(out, gamma)
            sev = (gamma - 1.8) / (3.5 - 1.8)
        elif issue == "overexposure":
            gamma = random.uniform(0.2, 0.5)
            out = apply_exposure(out, gamma)
            sev = (0.5 - gamma) / (0.5 - 0.2)
        elif issue == "noise":
            sigma = random.uniform(10, 50)
            out = apply_noise(out, sigma)
            sev = min(sigma / 50.0, 1.0)
        elif issue == "corruption":
            quality = random.randint(5, 30)
            out = apply_corruption(out, quality)
            sev = min((30 - quality) / 25.0, 1.0)

        idx = ISSUE_NAMES.index(issue)
        label[idx] = 1.0
        severities[issue] = float(sev)

    penalty = sum(severities[i] * ISSUE_PENALTY[i] for i in severities)
    score = float(np.clip(100.0 - penalty, 0, 100))
    return out, label, severities, score

In [24]:
def build_split(paths, split_name, variants_per_image=3, img_size=400):
    out_dir = os.path.join(ROOT, split_name)
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    counter = 0
    for p in tqdm(paths, desc=f"Building {split_name}"):
        img = cv2.imread(p)
        if img is None:
            continue
        img = resize_with_padding(img, img_size)
        for v in range(variants_per_image):
            deg_img, label, sev, score = generate_variant(img)
            fname = f"{split_name}_{counter:06d}.jpg"
            fpath = os.path.join(out_dir, fname)
            cv2.imwrite(fpath, deg_img)
            row = {"path": fpath, "quality_score": score}
            for i, name in enumerate(ISSUE_NAMES):
                row[name] = label[i]
            rows.append(row)
            counter += 1
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(ROOT, f"{split_name}_labels.csv"), index=False)
    return df

In [25]:
train_df = build_split(train_paths, "train", variants_per_image=3)
val_df = build_split(val_paths, "val", variants_per_image=2)
test_df = build_split(test_paths, "test", variants_per_image=2)

print(train_df.shape, val_df.shape, test_df.shape)

Building test: 100%|██████████| 300/300 [00:05<00:00, 52.17it/s]

(4200, 7) (600, 7) (600, 7)


In [26]:
AE_SIZE = 256

ae_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((AE_SIZE, AE_SIZE)),
    transforms.ToTensor(),
])

class CleanImageDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return ae_transform(img)

clean_train_ds = CleanImageDataset(train_paths)
clean_val_ds   = CleanImageDataset(val_paths)
clean_train_loader = DataLoader(clean_train_ds, batch_size=32, shuffle=True, num_workers=2)
clean_val_loader   = DataLoader(clean_val_ds, batch_size=32, shuffle=False, num_workers=2)

In [27]:
def gradient_loss(recon, target):
    recon_dx = recon[:, :, :, 1:] - recon[:, :, :, :-1]
    recon_dy = recon[:, :, 1:, :] - recon[:, :, :-1, :]

    target_dx = target[:, :, :, 1:] - target[:, :, :, :-1]
    target_dy = target[:, :, 1:, :] - target[:, :, :-1, :]

    loss_x = F.l1_loss(recon_dx, target_dx)
    loss_y = F.l1_loss(recon_dy, target_dy)

    return loss_x + loss_y


def ae_loss(recon, target):
    mse = F.mse_loss(recon, target)

    ssim_loss = 1 - ssim(
        recon,
        target,
        data_range=1.0,
        size_average=True
    )

    grad = gradient_loss(recon, target)

    return (
        0.55 * mse +
        0.30 * ssim_loss +
        0.15 * grad
    )

In [44]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 3, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.dec = nn.Sequential(
            nn.ConvTranspose2d(
                256, 128, 4, 2, 1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(
                128, 64, 4, 2, 1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(
                64, 32, 4, 2, 1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(
                32, 3, 4, 2, 1
            ),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.enc(x)
        return self.dec(z)

autoencoder = ConvAutoencoder().to(device)
ae_optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
ae_criterion = nn.MSELoss()

AE_EPOCHS = 30
best_val_loss = float("inf")

for epoch in range(AE_EPOCHS):
    autoencoder.train()
    train_loss = 0.0
    for clean_img in clean_train_loader:
        clean_img = clean_img.to(device)
        recon = autoencoder(clean_img)
        loss = ae_loss(recon, clean_img)
        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        train_loss += loss.item() * clean_img.size(0)
    train_loss /= len(clean_train_ds)

    autoencoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for clean_img in clean_val_loader:
            clean_img = clean_img.to(device)
            recon = autoencoder(clean_img)
            val_loss += ae_loss(recon, clean_img).item() * clean_img.size(0)
    val_loss /= len(clean_val_ds)

    print(f"[AE] epoch {epoch+1}/{AE_EPOCHS} train_loss={train_loss:.5f} val_loss={val_loss:.5f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(autoencoder.state_dict(), os.path.join(ROOT, "autoencoder.pt"))

autoencoder.load_state_dict(torch.load(os.path.join(ROOT, "autoencoder.pt")))
autoencoder.eval()
for p in autoencoder.parameters():
    p.requires_grad = False
print("Autoencoder frozen and ready.")

[AE] epoch 1/30 train_loss=0.24441 val_loss=0.23575
[AE] epoch 2/30 train_loss=0.20202 val_loss=0.19857
[AE] epoch 3/30 train_loss=0.16546 val_loss=0.16058
[AE] epoch 4/30 train_loss=0.14095 val_loss=0.13577
[AE] epoch 5/30 train_loss=0.12509 val_loss=0.11821
[AE] epoch 6/30 train_loss=0.11298 val_loss=0.11738
[AE] epoch 7/30 train_loss=0.10520 val_loss=0.10640
[AE] epoch 8/30 train_loss=0.09907 val_loss=0.09543
[AE] epoch 9/30 train_loss=0.09333 val_loss=0.09606
[AE] epoch 10/30 train_loss=0.08942 val_loss=0.09237
[AE] epoch 11/30 train_loss=0.08598 val_loss=0.08556
[AE] epoch 12/30 train_loss=0.08390 val_loss=0.08363
[AE] epoch 13/30 train_loss=0.08023 val_loss=0.08779
[AE] epoch 14/30 train_loss=0.07859 val_loss=0.07787
[AE] epoch 15/30 train_loss=0.07828 val_loss=0.08180
[AE] epoch 16/30 train_loss=0.07523 val_loss=0.07400
[AE] epoch 17/30 train_loss=0.07422 val_loss=0.07420
[AE] epoch 18/30 train_loss=0.07290 val_loss=0.07475
[AE] epoch 19/30 train_loss=0.07182 val_loss=0.07269
[A

In [45]:
@torch.no_grad()
def extract_anomaly_feats(model, img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tensor = ae_transform(img_rgb).unsqueeze(0).to(device)
    recon = model(tensor)
    err_map = (
        tensor - recon
    ).pow(2).mean(dim=1).squeeze(0).cpu().numpy()

    h, w = err_map.shape
    grid_errors = []

    for gy in range(8):
        for gx in range(8):

            y1 = gy * h // 8
            y2 = (gy + 1) * h // 8

            x1 = gx * w // 8
            x2 = (gx + 1) * w // 8

            patch = err_map[y1:y2, x1:x2]

            grid_errors.append(patch.mean())

    grid_errors = np.array(grid_errors)
    local_max_err = grid_errors.max()
    local_top3_mean = np.mean(
        np.sort(grid_errors)[-3:]
    )

    local_std_err = grid_errors.std()
    mean_err = err_map.mean()
    max_err = err_map.max()
    std_err = err_map.std()

    p90 = np.percentile(err_map, 90)
    p99 = np.percentile(err_map, 99)

    pct_anomalous = np.mean(
        err_map > (mean_err + 2 * std_err)
    )

    gray_input = (
        0.299 * tensor[:, 0] +
        0.587 * tensor[:, 1] +
        0.114 * tensor[:, 2]
    ).unsqueeze(1)

    gray_recon = (
        0.299 * recon[:, 0] +
        0.587 * recon[:, 1] +
        0.114 * recon[:, 2]
    ).unsqueeze(1)

    input_dx = gray_input[:, :, :, 1:] - gray_input[:, :, :, :-1]
    input_dy = gray_input[:, :, 1:, :] - gray_input[:, :, :-1, :]

    recon_dx = gray_recon[:, :, :, 1:] - gray_recon[:, :, :, :-1]
    recon_dy = gray_recon[:, :, 1:, :] - gray_recon[:, :, :-1, :]

    grad_error_x = torch.abs(input_dx - recon_dx)
    grad_error_y = torch.abs(input_dy - recon_dy)

    grad_error = torch.cat([
        grad_error_x.flatten(),
        grad_error_y.flatten()
    ])

    mean_grad_err = grad_error.mean().item()
    max_grad_err = grad_error.max().item()
    p95_grad_err = torch.quantile(
        grad_error,
        0.95
    ).item()

    # Slice to common dimensions (H-1, W-1) to avoid shape mismatch
    input_edges = torch.sqrt(
        input_dx[:, :, :-1, :].pow(2) + input_dy[:, :, :, :-1].pow(2) + 1e-8
    )

    mean_edge_strength = input_edges.mean().item()

    return np.array([
        mean_err,
        max_err,
        std_err,
        p90,
        p99,
        pct_anomalous,
        mean_grad_err,
        max_grad_err,
        p95_grad_err,
        local_max_err,
        local_std_err
    ], dtype=np.float32), err_map

ANOMALY_FEAT_NAMES = [
    "mean_err",
    "max_err",
    "std_err",
    "p90_err",
    "p99_err",
    "pct_anomalous",
    "mean_grad_err",
    "max_grad_err",
    "p95_grad_err",
    "local_max_err",
    "local_std_err"
]

In [46]:
def precompute_features(df, split_name):
    classical_list, anomaly_list = [], []
    for p in tqdm(df["path"], desc=f"Features {split_name}"):
        img = cv2.imread(p)
        classical_list.append(compute_classical_features(img))
        a_feats, _ = extract_anomaly_feats(autoencoder, img)
        anomaly_list.append(a_feats)

    classical_arr = np.stack(classical_list)
    anomaly_arr = np.stack(anomaly_list)

    np.save(os.path.join(ROOT, f"{split_name}_classical.npy"), classical_arr)
    np.save(os.path.join(ROOT, f"{split_name}_anomaly.npy"), anomaly_arr)
    return classical_arr, anomaly_arr

train_classical, train_anomaly = precompute_features(train_df, "train")
val_classical,   val_anomaly   = precompute_features(val_df,   "val")
test_classical,  test_anomaly  = precompute_features(test_df,  "test")

# Normalize classical/anomaly features using train-set statistics
classical_mean, classical_std = train_classical.mean(0), train_classical.std(0) + 1e-6
anomaly_mean, anomaly_std     = train_anomaly.mean(0), train_anomaly.std(0) + 1e-6

def normalize(arr, mean, std):
    return (arr - mean) / std

train_classical_n = normalize(train_classical, classical_mean, classical_std)
val_classical_n   = normalize(val_classical, classical_mean, classical_std)
test_classical_n  = normalize(test_classical, classical_mean, classical_std)

train_anomaly_n = normalize(train_anomaly, anomaly_mean, anomaly_std)
val_anomaly_n   = normalize(val_anomaly, anomaly_mean, anomaly_std)
test_anomaly_n  = normalize(test_anomaly, anomaly_mean, anomaly_std)

np.savez(os.path.join(ROOT, "norm_stats.npz"),
         classical_mean=classical_mean, classical_std=classical_std,
         anomaly_mean=anomaly_mean, anomaly_std=anomaly_std)

Features test: 100%|██████████| 600/600 [00:15<00:00, 38.02it/s]


In [47]:
IMG_SIZE = 224

cnn_transform_train = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

cnn_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FusionDataset(Dataset):
    def __init__(self, df, classical_n, anomaly_n, transform):
        self.df = df.reset_index(drop=True)
        self.classical_n = classical_n
        self.anomaly_n = anomaly_n
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_t = self.transform(img)

        classical_t = torch.tensor(self.classical_n[idx], dtype=torch.float32)
        anomaly_t   = torch.tensor(self.anomaly_n[idx], dtype=torch.float32)
        issue_labels = torch.tensor(row[ISSUE_NAMES].values.astype(np.float32))
        score = torch.tensor([row["quality_score"] / 100.0], dtype=torch.float32)  # scaled 0-1

        return img_t, classical_t, anomaly_t, issue_labels, score

train_fusion_ds = FusionDataset(train_df, train_classical_n, train_anomaly_n, cnn_transform_train)
val_fusion_ds   = FusionDataset(val_df, val_classical_n, val_anomaly_n, cnn_transform)
test_fusion_ds  = FusionDataset(test_df, test_classical_n, test_anomaly_n, cnn_transform)

train_fusion_loader = DataLoader(train_fusion_ds, batch_size=32, shuffle=True, num_workers=2)
val_fusion_loader   = DataLoader(val_fusion_ds, batch_size=32, shuffle=False, num_workers=2)
test_fusion_loader  = DataLoader(test_fusion_ds, batch_size=32, shuffle=False, num_workers=2)


In [48]:
class FusedQualityModel(nn.Module):
    def __init__(self, n_issues=5, n_engineered=16, n_anomaly_feats=10):
        super().__init__()
        backbone = mobilenet_v2(weights="IMAGENET1K_V2")
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.cnn_norm = nn.LayerNorm(1280)
        fusion_dim = 1280 + n_engineered + n_anomaly_feats
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU()
        )
        self.issue_out = nn.Linear(64, n_issues)
        self.defect_out = nn.Linear(64, 1)
        self.score_out = nn.Linear(64, 1)
        self.classical_direct = nn.Linear(
            n_engineered,
            n_issues,
            bias=True
        )
        self.classical_direct_scale = 0.5

    def forward(self, img, engineered_feats, anomaly_feats):
        cnn_emb = self.cnn_norm(self.pool(self.features(img)).flatten(1))
        x = torch.cat([cnn_emb, engineered_feats, anomaly_feats], dim=1)
        x = self.head(x)
        issue_logits = (
            self.issue_out(x)
            + self.classical_direct_scale * self.classical_direct(engineered_feats)
        )
        score = torch.sigmoid(self.score_out(x))
        return issue_logits, self.defect_out(x), score

model = FusedQualityModel(n_issues=len(ISSUE_NAMES), n_engineered=16, n_anomaly_feats=10).to(device)
pos_weight = torch.tensor([1.0, 1.3, 1.3, 1.0, 1.0], device=device)
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
mse_loss = nn.MSELoss()

In [49]:
def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for img, classical, anomaly, issue_labels, score in loader:
            img, classical, anomaly = img.to(device), classical.to(device), anomaly.to(device)
            issue_labels, score = issue_labels.to(device), score.to(device)

            issue_logits, defect_logit, score_pred = model(img, classical, anomaly)

            defect_label = (issue_labels.sum(dim=1) > 0).float().unsqueeze(1)
            loss_issue  = bce_loss(issue_logits, issue_labels)
            loss_defect = F.binary_cross_entropy_with_logits(defect_logit, defect_label)
            loss_score  = mse_loss(score_pred, score)
            loss = 1.0 * loss_issue + 0.3 * loss_defect + 1.2 * loss_score

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * img.size(0)
    return total_loss / len(loader.dataset)

# Re-initialize model with correct feature count (11 anomaly features)
model = FusedQualityModel(n_issues=len(ISSUE_NAMES), n_engineered=16, n_anomaly_feats=11).to(device)

# Phase 2a — freeze CNN backbone, train the head
for p in model.features.parameters():
    p.requires_grad = False

optimizer_a = torch.optim.Adam(
    list(model.head.parameters()) + list(model.issue_out.parameters()) +
    list(model.defect_out.parameters()) + list(model.score_out.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

WARMUP_EPOCHS = 5
for epoch in range(WARMUP_EPOCHS):
    tr_loss = run_epoch(train_fusion_loader, model, optimizer_a)
    val_loss = run_epoch(val_fusion_loader, model)
    print(f"[Phase2a] epoch {epoch+1}/{WARMUP_EPOCHS} train_loss={tr_loss:.4f} val_loss={val_loss:.4f}")

# Phase 2b — unfreeze last few backbone blocks, fine-tune at low LR
for p in model.features[-4:].parameters():
    p.requires_grad = True
optimizer_b = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=1e-4)

scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

FINETUNE_EPOCHS = 12
best_val_loss = float("inf")
wait = 0
patience = 6
for epoch in range(FINETUNE_EPOCHS):
    tr_loss = run_epoch(train_fusion_loader, model, optimizer_b)
    val_loss = run_epoch(val_fusion_loader, model)
    scheduler_b.step(val_loss)

    print(f"[Phase2b] epoch {epoch+1}/{FINETUNE_EPOCHS} train_loss={tr_loss:.4f} val_loss={val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(ROOT, "fusion_model.pt"))
    else:
        wait += 1

        if wait >= patience:
            print("Early stopping")
            break

model.load_state_dict(torch.load(os.path.join(ROOT, "fusion_model.pt")))
model.eval()
print("Fusion model trained and saved.")

[Phase2a] epoch 1/5 train_loss=0.5024 val_loss=0.4051
[Phase2a] epoch 2/5 train_loss=0.3506 val_loss=0.3784
[Phase2a] epoch 3/5 train_loss=0.3027 val_loss=0.3716
[Phase2a] epoch 4/5 train_loss=0.2711 val_loss=0.3925
[Phase2a] epoch 5/5 train_loss=0.2529 val_loss=0.3984
[Phase2b] epoch 1/12 train_loss=0.2068 val_loss=0.3900
[Phase2b] epoch 2/12 train_loss=0.1993 val_loss=0.3831
[Phase2b] epoch 3/12 train_loss=0.1869 val_loss=0.3811
[Phase2b] epoch 4/12 train_loss=0.1876 val_loss=0.3759
[Phase2b] epoch 5/12 train_loss=0.1811 val_loss=0.3760
[Phase2b] epoch 6/12 train_loss=0.1824 val_loss=0.3735
[Phase2b] epoch 7/12 train_loss=0.1748 val_loss=0.3736
[Phase2b] epoch 8/12 train_loss=0.1708 val_loss=0.3714
[Phase2b] epoch 9/12 train_loss=0.1689 val_loss=0.3739
[Phase2b] epoch 10/12 train_loss=0.1634 val_loss=0.3771
[Phase2b] epoch 11/12 train_loss=0.1637 val_loss=0.3717
[Phase2b] epoch 12/12 train_loss=0.1564 val_loss=0.3722
Early stopping
Fusion model trained and saved.


In [50]:
@torch.no_grad()
def collect_predictions(loader, model):
    all_issue_probs, all_issue_true = [], []
    all_score_pred, all_score_true = [], []
    for img, classical, anomaly, issue_labels, score in loader:
        img, classical, anomaly = img.to(device), classical.to(device), anomaly.to(device)
        issue_logits, defect_logit, score_pred = model(img, classical, anomaly)
        all_issue_probs.append(torch.sigmoid(issue_logits).cpu().numpy())
        all_issue_true.append(issue_labels.numpy())
        all_score_pred.append(score_pred.cpu().numpy())
        all_score_true.append(score.numpy())
    return (np.concatenate(all_issue_probs), np.concatenate(all_issue_true),
            np.concatenate(all_score_pred), np.concatenate(all_score_true))

issue_probs, issue_true, score_pred, score_true = collect_predictions(test_fusion_loader, model)
issue_pred_bin = (issue_probs > 0.5).astype(int)

print("\n=== Per-issue metrics (test set) ===")
for i, name in enumerate(ISSUE_NAMES):
    p, r, f1, _ = precision_recall_fscore_support(
        issue_true[:, i], issue_pred_bin[:, i], average="binary", zero_division=0)
    try:
        auc = roc_auc_score(issue_true[:, i], issue_probs[:, i])
    except ValueError:
        auc = float("nan")  # happens if a class is missing in a small test split
    cm = confusion_matrix(issue_true[:, i], issue_pred_bin[:, i])
    print(f"{name:15s} precision={p:.3f} recall={r:.3f} f1={f1:.3f} auc={auc:.3f}")
    print(f"  confusion_matrix:\n{cm}")

print("\n=== Threshold sweep for underexposure / overexposure ===")
for i, name in enumerate(ISSUE_NAMES):
    if name not in ("underexposure", "overexposure"):
        continue
    for thresh in [0.3, 0.35, 0.4, 0.45, 0.5]:
        pred = (issue_probs[:, i] > thresh).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(
            issue_true[:, i], pred, average="binary", zero_division=0)
        print(f"{name:15s} thresh={thresh:.2f} precision={p:.3f} recall={r:.3f} f1={f1:.3f}")

score_pred_clipped = np.clip(score_pred, 0, 1)
mae = mean_absolute_error(score_true * 100, score_pred_clipped * 100)
rmse = math.sqrt(mean_squared_error(score_true * 100, score_pred_clipped * 100))
print(f"\nQuality score regression — MAE={mae:.2f} RMSE={rmse:.2f}")

# Failure case inspection — worst-scoring predictions
errors = np.abs(score_true.flatten() - score_pred_clipped.flatten())
worst_idx = np.argsort(-errors)[:10]

print("\nWorst 10 quality-score predictions (row index into test_df):")
print(test_df.iloc[worst_idx][["path", "quality_score"]].assign(predicted=score_pred_clipped[worst_idx]*100))


=== Per-issue metrics (test set) ===
blur            precision=0.971 recall=0.930 f1=0.950 auc=0.995
  confusion_matrix:
[[380   6]
 [ 15 199]]
underexposure   precision=0.733 recall=0.644 f1=0.686 auc=0.835
  confusion_matrix:
[[326  52]
 [ 79 143]]
overexposure    precision=0.793 recall=0.713 f1=0.751 auc=0.878
  confusion_matrix:
[[352  39]
 [ 60 149]]
noise           precision=0.927 recall=0.955 f1=0.941 auc=0.993
  confusion_matrix:
[[385  15]
 [  9 191]]
corruption      precision=0.898 recall=0.815 f1=0.854 auc=0.959
  confusion_matrix:
[[364  20]
 [ 40 176]]

=== Threshold sweep for underexposure / overexposure ===
underexposure   thresh=0.30 precision=0.634 recall=0.757 f1=0.690
underexposure   thresh=0.35 precision=0.664 recall=0.730 f1=0.695
underexposure   thresh=0.40 precision=0.703 recall=0.703 f1=0.703
underexposure   thresh=0.45 precision=0.722 recall=0.667 f1=0.693
underexposure   thresh=0.50 precision=0.733 recall=0.644 f1=0.686
overexposure    thresh=0.30 precision=0

In [51]:
clean_test_imgs = [resize_with_padding(cv2.imread(p), IMG_SIZE) for p in test_paths]
random.seed(123)
np.random.seed(123)

defect_test_imgs = [
    apply_ood_defect(img)
    for img in clean_test_imgs
]

def anomaly_ensemble_score(raw_feats, feat_mean, feat_std):
    z = (raw_feats - feat_mean) / (feat_std + 1e-8)
    return z.max()

def mean_recon_error_ensemble(imgs):
    errs = []
    for img in imgs:
        feats, _ = extract_anomaly_feats(autoencoder, img)
        errs.append(anomaly_ensemble_score(feats, anomaly_mean, anomaly_std))
    return np.array(errs)

clean_errs_ens  = mean_recon_error_ensemble(clean_test_imgs)
defect_errs_ens = mean_recon_error_ensemble(defect_test_imgs)
y_true = np.concatenate([np.zeros_like(clean_errs_ens), np.ones_like(defect_errs_ens)])
y_score = np.concatenate([clean_errs_ens, defect_errs_ens])
print(f"Ensemble anomaly AUC: {roc_auc_score(y_true, y_score):.3f}")

anomaly_auc = roc_auc_score(y_true, y_score)
print(f"\nAnomaly branch ROC-AUC (clean vs synthetic-defect): {anomaly_auc:.3f}")
print(f"Mean recon error — clean: {clean_errs_ens.mean():.5f} | defect: {defect_errs_ens.mean():.5f}")

print("\n=== Per-anomaly-feature AUC breakdown ===")
clean_feats_all  = np.stack([extract_anomaly_feats(autoencoder, img)[0] for img in clean_test_imgs])
defect_feats_all = np.stack([extract_anomaly_feats(autoencoder, img)[0] for img in defect_test_imgs])
y_true_all = np.concatenate([np.zeros(len(clean_feats_all)), np.ones(len(defect_feats_all))])

feat_aucs = []
for i, name in enumerate(ANOMALY_FEAT_NAMES):
    y_score = np.concatenate([clean_feats_all[:, i], defect_feats_all[:, i]])
    auc = roc_auc_score(y_true_all, y_score)
    feat_aucs.append(auc)
    print(f"{name:16s} AUC={auc:.3f}")

best_idx = int(np.argmax(feat_aucs))
print(f"Best single feature: {ANOMALY_FEAT_NAMES[best_idx]} (index {best_idx})")

Ensemble anomaly AUC: 0.711

Anomaly branch ROC-AUC (clean vs synthetic-defect): 0.711
Mean recon error — clean: 1.84531 | defect: 3.28857

=== Per-anomaly-feature AUC breakdown ===
mean_err         AUC=0.648
max_err          AUC=0.674
std_err          AUC=0.682
p90_err          AUC=0.606
p99_err          AUC=0.675
pct_anomalous    AUC=0.482
mean_grad_err    AUC=0.589
max_grad_err     AUC=0.653
p95_grad_err     AUC=0.594
local_max_err    AUC=0.670
local_std_err    AUC=0.649
Best single feature: std_err (index 2)


In [52]:
model.eval()
dummy_img = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
dummy_classical = torch.randn(1, 16).to(device)
dummy_anomaly = torch.randn(1, 11).to(device)

torch.onnx.export(
    model, (dummy_img, dummy_classical, dummy_anomaly),
    os.path.join(ROOT, "fusion_model.onnx"),
    input_names=["image", "classical_feats", "anomaly_feats"],
    output_names=["issue_logits", "defect_logit", "score_pred"],
    dynamic_axes={"image": {0: "batch"}, "classical_feats": {0: "batch"},
                  "anomaly_feats": {0: "batch"}},
    opset_version=18
)

autoencoder.eval()
dummy_ae_input = torch.randn(1, 3, AE_SIZE, AE_SIZE).to(device)
torch.onnx.export(
    autoencoder, dummy_ae_input,
    os.path.join(ROOT, "autoencoder.onnx"),
    input_names=["image"], output_names=["reconstruction"],
    dynamic_axes={"image": {0: "batch"}, "reconstruction": {0: "batch"}},
    opset_version=18
)

print("Exported: fusion_model.onnx, autoencoder.onnx")
print("Also saved: norm_stats.npz — your backend MUST apply the same")
print("classical/anomaly normalization (mean/std) before calling the ONNX model.")

/tmp/ipykernel_1988/685964971.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `FusedQualityModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `FusedQualityModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: batch will not be used, since it shares the same shape constraints with another axis: batch.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/tmp/ipykernel_1988/685964971.py:18: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ConvAutoencoder([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ConvAutoencoder([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported: fusion_model.onnx, autoencoder.onnx
Also saved: norm_stats.npz — your backend MUST apply the same
classical/anomaly normalization (mean/std) before calling the ONNX model.


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [56]:
def apply_specific_ood_defect(img, defect_type):
    out = img.copy()
    h, w = out.shape[:2]

    if defect_type == "block":
        x1 = random.randint(0, w - 30)
        y1 = random.randint(0, h - 30)
        x2 = min(w - 1, x1 + random.randint(20, 80))
        y2 = min(h - 1, y1 + random.randint(20, 80))

        color = tuple(
            int(c) for c in np.random.randint(0, 255, 3)
        )
        cv2.rectangle(out, (x1, y1), (x2, y2), color, -1)

    elif defect_type == "line":
        x1 = random.randint(0, w - 1)
        y1 = random.randint(0, h - 1)

        x2 = np.clip(
            x1 + random.randint(-100, 100),
            0, w - 1
        )
        y2 = np.clip(
            y1 + random.randint(-100, 100),
            0, h - 1
        )

        cv2.line(
            out,
            (x1, y1),
            (x2, y2),
            (0, 0, 0),
            thickness=random.randint(5, 10)
        )

    elif defect_type == "checkerboard":
        size = random.randint(40, 90)
        x = random.randint(0, w - size)
        y = random.randint(0, h - size)

        patch = out[y:y+size, x:x+size].copy()
        tile = 8

        for i in range(0, size, tile):
            for j in range(0, size, tile):
                if (i // tile + j // tile) % 2 == 0:
                    patch[i:i+tile, j:j+tile] = 255
                else:
                    patch[i:i+tile, j:j+tile] = 0

        out[y:y+size, x:x+size] = patch

    elif defect_type == "channel_shift":
        channel = random.randint(0, 2)
        amount = random.randint(40, 120)

        out[:, :, channel] = np.clip(
            out[:, :, channel].astype(np.int16) + amount,
            0,
            255
        ).astype(np.uint8)

    elif defect_type == "local_blur":
        size = random.randint(40, 100)
        x = random.randint(0, w - size)
        y = random.randint(0, h - size)

        patch = out[y:y+size, x:x+size]

        out[y:y+size, x:x+size] = cv2.GaussianBlur(
            patch,
            (0, 0),
            random.uniform(3, 8)
        )

    elif defect_type == "local_noise":
        size = random.randint(40, 100)
        x = random.randint(0, w - size)
        y = random.randint(0, h - size)

        patch = out[y:y+size, x:x+size]

        noise = np.random.normal(
            0,
            random.uniform(30, 70),
            patch.shape
        )

        out[y:y+size, x:x+size] = np.clip(
            patch.astype(np.float32) + noise,
            0,
            255
        ).astype(np.uint8)

    elif defect_type == "scratch":
        x1 = random.randint(0, w - 1)
        y1 = random.randint(0, h - 1)

        length = random.randint(20, 100)
        angle = random.uniform(0, 2 * np.pi)

        x2 = int(np.clip(
            x1 + length * np.cos(angle),
            0, w - 1
        ))

        y2 = int(np.clip(
            y1 + length * np.sin(angle),
            0, h - 1
        ))

        value = random.choice([
            random.randint(180, 255),
            random.randint(0, 70)
        ])

        cv2.line(
            out,
            (x1, y1),
            (x2, y2),
            (value, value, value),
            thickness=random.randint(4, 6)
        )

    elif defect_type == "dead_pixel":
        for _ in range(random.randint(10, 40)):
            x = random.randint(0, w - 1)
            y = random.randint(0, h - 1)

            color = random.choice([
                (0, 0, 0),
                (255, 255, 255),
                tuple(
                    int(v)
                    for v in np.random.randint(0, 256, 3)
                )
            ])

            if random.random() < 0.4:
                cv2.circle(
                    out,
                    (x, y),
                    random.randint(1, 2),
                    color,
                    -1
                )
            else:
                out[y, x] = color

    return out

In [57]:
def evaluate_defect_type(defect_type):

    random.seed(123)
    np.random.seed(123)

    clean_imgs = [
        resize_with_padding(cv2.imread(p), IMG_SIZE)
        for p in test_paths
    ]

    defect_imgs = [
        apply_specific_ood_defect(img, defect_type)
        for img in clean_imgs
    ]

    clean_feats = np.stack([
        extract_anomaly_feats(autoencoder, img)[0]
        for img in clean_imgs
    ])

    defect_feats = np.stack([
        extract_anomaly_feats(autoencoder, img)[0]
        for img in defect_imgs
    ])

    y = np.concatenate([
        np.zeros(len(clean_feats)),
        np.ones(len(defect_feats))
    ])

    results = {}

    for i, name in enumerate(ANOMALY_FEAT_NAMES):
        scores = np.concatenate([
            clean_feats[:, i],
            defect_feats[:, i]
        ])

        results[name] = roc_auc_score(y, scores)

    return results

In [58]:
defect_types = [
    "block",
    "line",
    "checkerboard",
    "channel_shift",
    "local_noise",
    "scratch",
    "dead_pixel"
]

for defect_type in defect_types:
    results = evaluate_defect_type(defect_type)

    print(f"\n{defect_type}")
    for name, auc in results.items():
        print(f"  {name:16s}: {auc:.3f}")


block
  mean_err        : 0.708
  max_err         : 0.520
  std_err         : 0.745
  p90_err         : 0.637
  p99_err         : 0.745
  pct_anomalous   : 0.703
  mean_grad_err   : 0.484
  max_grad_err    : 0.499
  p95_grad_err    : 0.492
  local_max_err   : 0.801
  local_std_err   : 0.779

line
  mean_err        : 0.501
  max_err         : 0.537
  std_err         : 0.514
  p90_err         : 0.501
  p99_err         : 0.509
  pct_anomalous   : 0.489
  mean_grad_err   : 0.510
  max_grad_err    : 0.532
  p95_grad_err    : 0.512
  local_max_err   : 0.503
  local_std_err   : 0.500

checkerboard
  mean_err        : 0.742
  max_err         : 0.979
  std_err         : 0.915
  p90_err         : 0.622
  p99_err         : 0.902
  pct_anomalous   : 0.263
  mean_grad_err   : 0.727
  max_grad_err    : 0.955
  p95_grad_err    : 0.736
  local_max_err   : 0.863
  local_std_err   : 0.835

channel_shift
  mean_err        : 0.966
  max_err         : 0.435
  std_err         : 0.665
  p90_err         : 0.